<a href="https://colab.research.google.com/github/samreenfathima18/Guvi-Final-project/blob/main/Inventory_Optimization_%26_Business_Recommendation_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Import Libraries**

In [ ]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import os

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


**Load the Trained LightGBM Model**

In [ ]:
MODEL_PATH = "/content/drive/MyDrive/Enterprise_Retail_Intelligence/models/"

lgb_model = joblib.load(MODEL_PATH + "LightGBM.pkl")

print("✅ LightGBM model loaded successfully!")

✅ LightGBM model loaded successfully!


**Load the Prepared Test Data**

In [ ]:
DATA_PATH = "/content/drive/MyDrive/Enterprise_Retail_Intelligence/prepared_data/"

X_test = pd.read_parquet(DATA_PATH + "X_test.parquet")
y_test = pd.read_parquet(DATA_PATH + "y_test.parquet")["sales"]

print(X_test.shape)
print(y_test.shape)

(874911, 36)
(874911,)


In [ ]:
X_test = X_test.drop(columns=["date"])

**Make Predictions**

In [ ]:
test_pred = lgb_model.predict(X_test)

print(test_pred[:10])

[0.24922342 6.8928328  0.86908813 0.53424846 0.35418495 3.48626568
 5.07561866 0.2932075  1.64291322 2.17756044]


**Create Prediction DataFrame**

In [ ]:
prediction_df = pd.DataFrame({
    "Actual_Sales": y_test.values,
    "Predicted_Sales": test_pred
})

prediction_df.head()

,Actual_Sales,Predicted_Sales
0,0,0.249223
1,7,6.892833
2,0,0.869088
3,1,0.534248
4,0,0.354185


**Inventory Recommendation**

In [ ]:
np.random.seed(42)

prediction_df["Current_Stock"] = np.random.randint(
    0,
    30,
    len(prediction_df)
)

In [ ]:
prediction_df["Recommended_Order"] = np.maximum(
    prediction_df["Predicted_Sales"] - prediction_df["Current_Stock"],
    0
)

**Inventory Status**

In [ ]:
prediction_df["Inventory_Status"] = np.where(
    prediction_df["Predicted_Sales"] > prediction_df["Current_Stock"],
    "Restock",
    "Sufficient Stock"
)

prediction_df.head()

,Actual_Sales,Predicted_Sales,Current_Stock,Recommended_Order,Inventory_Status
0,0,0.249223,6,0.0,Sufficient Stock
1,7,6.892833,19,0.0,Sufficient Stock
2,0,0.869088,28,0.0,Sufficient Stock
3,1,0.534248,14,0.0,Sufficient Stock
4,0,0.354185,10,0.0,Sufficient Stock


**Save the Recommendations**

In [ ]:
RESULTS_PATH = "/content/drive/MyDrive/Enterprise_Retail_Intelligence/results/"
os.makedirs(RESULTS_PATH, exist_ok=True)

prediction_df.to_csv(
    RESULTS_PATH + "inventory_recommendations.csv",
    index=False
)

print("✅ Inventory recommendations saved successfully!")

✅ Inventory recommendations saved successfully!
